# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

I use the February 2026 decision window so the signal audit only uses information available before the future outcome window. The analysis is based on the content/client feature frame used in the previous week's leakage check.

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Signal audit window: February 2026")

Connected to FlyRank warehouse.
Signal audit window: February 2026


In [15]:
# Check the columns in the dim_content parquet file
con.sql(f"DESCRIBE SELECT * FROM '{DIM_CONTENT}'").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 1. Distributions

I inspect the distributions of the main signals before choosing thresholds. Search-performance variables can be heavy-tailed, so I use descriptive statistics and quantiles rather than relying only on the mean.

In [17]:
signal_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0
            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )
            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.avg_position_feb,

    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        DATE '2026-02-28'
    ) AS days_since_last_update

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

WHERE
    f.gsc_impressions_feb >= 100
    AND f.gsc_clicks_feb >= 3
    AND c.is_published IS TRUE
    AND c.content_created_date <= DATE '2026-02-28'
""").df()

signals = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "avg_position_feb",
    "content_age_days",
    "days_since_last_update",
]

print("Rows:", len(signal_df))

display(
    signal_df[signals].describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]
    ).T
)

Rows: 29700


,count,mean,std,min,25%,50%,75%,90%,95%,max
gsc_impressions_feb,29700.0,4878.206195,7771.160910,100.00000,1322.750000,2561.000000,5370.000000,10649.100000,16541.550000,167303.000000
gsc_clicks_feb,29700.0,18.595084,46.995007,3.00000,4.000000,8.000000,17.000000,38.000000,63.000000,3310.000000
avg_position_feb,29700.0,7.771974,7.020005,0.03963,3.444262,5.422985,8.740648,17.439465,23.596124,69.695192
content_age_days,29700.0,182.278586,109.263573,3.00000,103.000000,184.000000,235.000000,344.000000,381.000000,463.000000
days_since_last_update,29700.0,-90.100034,44.892687,-128.00000,-124.000000,-107.000000,-81.000000,3.000000,3.000000,233.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test #1 — Content age

Hypothesis: older pages may have a different subsequent search-performance pattern than newer pages.

I compare March zero-click rates across content-age buckets using the February feature state. The result is descriptive and directional; it does not establish that age causes the outcome.

In [20]:
age_test = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar

    FROM read_parquet('{FACT}/month=2026-03/*.parquet')

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) < 90 THEN '<90 days'

        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) < 180 THEN '90-179 days'

        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) < 365 THEN '180-364 days'

        ELSE '365+ days'
    END AS age_bucket,

    COUNT(*) AS n,

    AVG(
        CASE
            WHEN COALESCE(m.clicks_mar, 0) = 0
            THEN 1.0
            ELSE 0.0
        END
    ) AS march_zero_click_rate

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

LEFT JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND c.is_published IS TRUE
    AND c.content_created_date <= DATE '2026-02-28'

GROUP BY 1

ORDER BY
    CASE
        WHEN age_bucket = '<90 days' THEN 1
        WHEN age_bucket = '90-179 days' THEN 2
        WHEN age_bucket = '180-364 days' THEN 3
        ELSE 4
    END
""").df()

display(age_test)

,age_bucket,n,march_zero_click_rate
0,<90 days,7415,0.040054
1,90-179 days,6520,0.063650
2,180-364 days,13416,0.051058
3,365+ days,2349,0.046403


### Verdict: MIXED

Content age shows a directional difference across buckets, but the pattern is not monotonic. The 90–179 day bucket has the highest observed March zero-click rate (6.37%), while the 365+ day bucket is lower (4.64%) than both 90–179 and 180–364 days. Therefore, content age alone is a mixed signal rather than a simple staleness rule.

### Signal test #2 — Search volume

Hypothesis: pages with higher measured search demand may represent a different opportunity level than pages with low search volume.

I compare March zero-click rates across February impression buckets. The buckets are based on the observed February distribution rather than the future outcome.

In [21]:
volume_test = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar

    FROM read_parquet('{FACT}/month=2026-03/*.parquet')

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN f.impressions_feb < 1000 THEN '<1k'

        WHEN f.impressions_feb < 2500 THEN '1k-2.5k'

        WHEN f.impressions_feb < 5000 THEN '2.5k-5k'

        WHEN f.impressions_feb < 10000 THEN '5k-10k'

        ELSE '10k+'
    END AS volume_bucket,

    COUNT(*) AS n,

    AVG(
        CASE
            WHEN COALESCE(m.clicks_mar, 0) = 0
            THEN 1.0
            ELSE 0.0
        END
    ) AS march_zero_click_rate

FROM feb f

LEFT JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3

GROUP BY 1

ORDER BY
    CASE
        WHEN volume_bucket = '<1k' THEN 1
        WHEN volume_bucket = '1k-2.5k' THEN 2
        WHEN volume_bucket = '2.5k-5k' THEN 3
        WHEN volume_bucket = '5k-10k' THEN 4
        ELSE 5
    END
""").df()

display(volume_test)

,volume_bucket,n,march_zero_click_rate
0,<1k,5200,0.143462
1,1k-2.5k,9417,0.054582
2,2.5k-5k,7083,0.027248
3,5k-10k,4751,0.011997
4,10k+,3278,0.006406


### Verdict: CONFIRMED

Search volume shows a strong directional relationship with the March outcome in this slice. The observed March zero-click rate decreases from 14.35% for pages with fewer than 1k February impressions to 0.64% for pages with 10k+ impressions. This supports using search volume as a prioritization signal, while remaining a measured association rather than a causal claim.

### Signal test #3 — Average position

Hypothesis: search visibility differs across position bands, so average position may provide a useful prioritization signal.

I compare March zero-click rates across February average-position buckets. Lower position numbers represent stronger observed search visibility.

In [22]:
position_test = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0

            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )

            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar

    FROM read_parquet('{FACT}/month=2026-03/*.parquet')

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN f.avg_position_feb <= 3 THEN '1-3'

        WHEN f.avg_position_feb <= 5 THEN '3-5'

        WHEN f.avg_position_feb <= 10 THEN '5-10'

        WHEN f.avg_position_feb <= 20 THEN '10-20'

        ELSE '20+'
    END AS position_bucket,

    COUNT(*) AS n,

    AVG(
        CASE
            WHEN COALESCE(m.clicks_mar, 0) = 0
            THEN 1.0
            ELSE 0.0
        END
    ) AS march_zero_click_rate

FROM feb f

LEFT JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND f.avg_position_feb IS NOT NULL

GROUP BY 1

ORDER BY
    CASE
        WHEN position_bucket = '1-3' THEN 1
        WHEN position_bucket = '3-5' THEN 2
        WHEN position_bucket = '5-10' THEN 3
        WHEN position_bucket = '10-20' THEN 4
        ELSE 5
    END
""").df()

display(position_test)

,position_bucket,n,march_zero_click_rate
0,1-3,5633,0.039411
1,3-5,7793,0.034646
2,5-10,9929,0.056400
3,10-20,4139,0.075622
4,20+,2235,0.074273


### Verdict: CONFIRMED

Average position shows a generally directional relationship with the March outcome. The observed zero-click rate is 3.46% for positions 3–5 and rises to 7.56% for positions 10–20, with 20+ at 7.43%. The small difference between the last two buckets means this is better treated as a directional visibility signal than an exact threshold.

## 3. The flag-linked test

The FlyRank refresh-oriented flags are connected to content staleness. I therefore use content age as the flag-linked signal and compare the measured March zero-click rate across February age bands.

The test checks whether the data provides directional support for using staleness as a refresh-review signal. It does not establish causation.

In [23]:
# Reuse the age signal test as the flag-linked test.
flag_linked_test = age_test.copy()

display(flag_linked_test)

print("\nFlag-linked signal: content age")
print("Buckets tested:", len(flag_linked_test))
print("Total rows:", int(flag_linked_test["n"].sum()))

,age_bucket,n,march_zero_click_rate
0,<90 days,7415,0.040054
1,90-179 days,6520,0.063650
2,180-364 days,13416,0.051058
3,365+ days,2349,0.046403



Flag-linked signal: content age
Buckets tested: 4
Total rows: 29700


### **Flag-linked verdict: MIXED**

Content age was tested as the flag-linked staleness signal. The observed March zero-click rates differ across age buckets, but they do not increase consistently with age. Therefore, this slice provides mixed evidence for a simple age-based refresh assumption and does not justify treating age alone as a reliable refresh trigger.

The measured February-to-March comparison suggests that **search volume provides a clear directional prioritization signal**, with the observed March zero-click rate decreasing from 14.35% for pages with fewer than 1k February impressions to 0.64% for pages with 10k+ impressions. **Average position also shows a generally directional pattern**, with zero-click rates increasing from 3.46% for positions 3–5 to 7.56% for positions 10–20. **Content age is mixed** as a refresh-related signal because its zero-click rates do not consistently increase across older age buckets.

For the baseline, I will use search volume and average position because their observed relationships are clear enough to justify a simple decision-support rule. The rule is a prioritization heuristic, not a causal claim.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.